# 04 - Prefill Conditions

For each condition, prefill Qwen3-4B with the (possibly transformed) COT text,
then sample the answer. This tests how much the model can recover from the COT alone.

Conditions:
- **self_prefill**: Original COT (should match normal accuracy)
- **paraphrase_light**: Light reword by Qwen3-8B
- **paraphrase_heavy**: Compressed to key steps by Qwen3-8B
- **shuffled_steps**: Steps randomly reordered
- **corrupted_numbers**: Intermediate numbers replaced with random values

The **normal** condition uses answers from `02_generate_cots.ipynb`.
The **no_cot** condition uses results from `02_generate_cots.ipynb`.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/11-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
from tqdm import tqdm
from vllm import LLM, SamplingParams
from lib.data import extract_predicted_answer
from lib.prompts import build_prefill_string

## Load COTs and Paraphrases

In [ ]:
# Load original COTs
cots = {}
for p in sorted(COT_CACHE.glob("*.json"), key=lambda x: int(x.stem)):
    data = json.loads(p.read_text())
    cots[data["problem_id"]] = data
print(f"Loaded {len(cots)} COTs")

# Load paraphrases
paraphrases = {}  # {condition: {problem_id: paraphrased_cot}}
for cond in ["paraphrase_light", "paraphrase_heavy", "shuffled_steps", "corrupted_numbers"]:
    paraphrases[cond] = {}
    for p in PARAPHRASE_CACHE.glob(f"{cond}_*.json"):
        data = json.loads(p.read_text())
        paraphrases[cond][data["problem_id"]] = data["paraphrased_cot"]
    print(f"Loaded {len(paraphrases[cond])} paraphrases for {cond}")

## Load Model

In [ ]:
llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    max_model_len=4096,
)
sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_ANSWER_TOKENS)
print(f"Loaded {MODEL_NAME}")

## Run All Prefill Conditions

In [ ]:
CHUNK_SIZE = 64
prefill_conditions = ["self_prefill", "paraphrase_light", "paraphrase_heavy", "shuffled_steps", "corrupted_numbers"]

for condition in prefill_conditions:
    # Resume check
    done_ids = {
        int(p.stem.split("_")[-1])
        for p in PREFILL_CACHE.glob(f"{condition}_*.json")
    }

    # Build the work list
    todo = []
    for pid, cot_data in cots.items():
        if pid in done_ids:
            continue
        if cot_data["error"] is not None or not cot_data["cot_text"]:
            continue

        if condition == "self_prefill":
            cot_text = cot_data["cot_text"]
        else:
            if pid not in paraphrases.get(condition, {}):
                continue
            cot_text = paraphrases[condition][pid]

        todo.append({
            "problem_id": pid,
            "question": cot_data["question"],
            "gold_answer": cot_data["gold_answer"],
            "cot_text": cot_text,
        })

    print(f"\n[{condition}] {len(done_ids)} done, {len(todo)} remaining")

    for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=condition):
        chunk = todo[i : i + CHUNK_SIZE]
        prompts = [build_prefill_string(ex["question"], ex["cot_text"]) for ex in chunk]
        outputs = llm.generate(prompts, sampling)

        for ex, output in zip(chunk, outputs):
            generated_text = output.outputs[0].text
            predicted = extract_predicted_answer(generated_text)

            result = {
                "problem_id": ex["problem_id"],
                "condition": condition,
                "question": ex["question"],
                "gold_answer": ex["gold_answer"],
                "cot_text": ex["cot_text"],
                "prefill_text": prompts[chunk.index(ex)],
                "generated_tokens": generated_text,
                "predicted_answer": predicted,
                "error": None,
            }
            (PREFILL_CACHE / f"{condition}_{ex['problem_id']}.json").write_text(
                json.dumps(result)
            )

    print(f"[{condition}] complete.")

print("\nAll prefill conditions complete.")

## Quick Accuracy Check

In [ ]:
for condition in ["no_cot"] + prefill_conditions:
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        results.append(json.loads(p.read_text()))

    if not results:
        print(f"{condition}: no results")
        continue

    correct = sum(1 for r in results if r["predicted_answer"] == r["gold_answer"])
    total = len(results)
    print(f"{condition:25s}: {correct}/{total} = {correct/total:.1%}")

In [ ]:
# Clean up
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("GPU memory freed. Ready for notebook 05.")